# Focal Loss & Class-Weighted CE Experiment for Bangla Punctuation Restoration

This notebook runs **three experiments** to address Reviewer C's class imbalance concern:

| Experiment | Loss Function | Output Directory |
|-----------|--------------|------------------|
| 1 (Baseline) | Standard Cross-Entropy | `out-ce/` |
| 2 | Focal Loss (γ=2.0) | `out-focal/` |
| 3 | Class-Weighted Cross-Entropy | `out-weighted-ce/` |

**Just run all cells top-to-bottom.** Cell 6 auto-generates a comparison table.

**Requirements:** Kaggle GPU runtime (T4 or P100). ~4.5 hours total.

## Cell 1: 📦 Setup & Dependencies

In [ ]:
# Install all dependencies in one clean command
# No virtualenv, no Python 3.8, no Rust needed — Kaggle's native kernel handles everything
!pip install -q transformers==4.39.3 tokenizers==0.15.2 pytorch-crf sentencepiece numpy==1.26.4 gdown

## Cell 2: 📂 Clone Repo & Prepare Workspace

In [ ]:
import os
import shutil

REPO_URL = "https://github.com/Obyedullahilmamun/Punctuation-Restoration-Bangla-Aug-Exp-2.git"
BRANCH = "focal-loss-experiment"
REPO_DIR = "Punctuation-Restoration-Bangla-Aug-Exp-2"
WEIGHTS_ID = "1X2udyT1XYrmCNvWtFpT_6jrWsQejGCBW"

# Clean previous runs
!rm -rf {REPO_DIR}

# Clone the focal-loss-experiment branch
!git clone -b {BRANCH} {REPO_URL}

# Setup directories
os.makedirs(f"{REPO_DIR}/weights", exist_ok=True)
os.makedirs(f"{REPO_DIR}/data/test", exist_ok=True)

# Copy test data splits
bn_data_dir = f"{REPO_DIR}/data/bn"
for file in os.listdir(bn_data_dir):
    if file.startswith("test_"):
        shutil.copy(os.path.join(bn_data_dir, file), f"{REPO_DIR}/data/test/{file}")
        print(f"  Copied {file} -> data/test/")

# Download pretrained weights
!gdown {WEIGHTS_ID} -O {REPO_DIR}/weights/xlm-roberta-large-bn.pt

print("\n✅ Workspace ready!")
print(f"  Repo: {REPO_DIR} (branch: {BRANCH})")
print(f"  Test files: {os.listdir(f'{REPO_DIR}/data/test')}")

## Cell 3: 🔬 Experiment 1 — Baseline (Standard Cross-Entropy)

In [ ]:
%%bash
cd Punctuation-Restoration-Bangla-Aug-Exp-2
export PYTHONWARNINGS="ignore"

echo "========================================"
echo "  EXPERIMENT 1: Standard Cross-Entropy"
echo "========================================"

# Training
python src/train.py \
    --cuda=True \
    --pretrained-model=xlm-roberta-large \
    --freeze-bert=False \
    --lstm-dim=-1 \
    --language=bangla \
    --seed=1 \
    --lr=5e-6 \
    --epoch=3 \
    --use-crf=False \
    --augment-type=all \
    --augment-rate=0.20 \
    --alpha-sub=0.4 \
    --alpha-del=0.4 \
    --loss-type=ce \
    --data-path=data \
    --save-path=out-ce

echo ""
echo "--- Standalone Evaluation (CE) ---"

# Copy test data for standalone evaluation
python src/test.py \
    --pretrained-model=xlm-roberta-large \
    --lstm-dim=-1 \
    --use-crf=False \
    --data-path=data/test \
    --weight-path=out-ce/weights.pt \
    --sequence-length=256 \
    --save-path=out-ce

echo ""
echo "✅ Experiment 1 complete. Results in out-ce/"

## Cell 4: 🔬 Experiment 2 — Focal Loss (γ=2.0)

In [ ]:
%%bash
cd Punctuation-Restoration-Bangla-Aug-Exp-2
export PYTHONWARNINGS="ignore"

echo "========================================"
echo "  EXPERIMENT 2: Focal Loss (gamma=2.0)"
echo "========================================"

# Training
python src/train.py \
    --cuda=True \
    --pretrained-model=xlm-roberta-large \
    --freeze-bert=False \
    --lstm-dim=-1 \
    --language=bangla \
    --seed=1 \
    --lr=5e-6 \
    --epoch=3 \
    --use-crf=False \
    --augment-type=all \
    --augment-rate=0.20 \
    --alpha-sub=0.4 \
    --alpha-del=0.4 \
    --loss-type=focal \
    --focal-gamma=2.0 \
    --data-path=data \
    --save-path=out-focal

echo ""
echo "--- Standalone Evaluation (Focal) ---"

python src/test.py \
    --pretrained-model=xlm-roberta-large \
    --lstm-dim=-1 \
    --use-crf=False \
    --data-path=data/test \
    --weight-path=out-focal/weights.pt \
    --sequence-length=256 \
    --save-path=out-focal

echo ""
echo "✅ Experiment 2 complete. Results in out-focal/"

## Cell 5: 🔬 Experiment 3 — Class-Weighted Cross-Entropy

In [ ]:
%%bash
cd Punctuation-Restoration-Bangla-Aug-Exp-2
export PYTHONWARNINGS="ignore"

echo "============================================="
echo "  EXPERIMENT 3: Class-Weighted Cross-Entropy"
echo "============================================="

# Training
python src/train.py \
    --cuda=True \
    --pretrained-model=xlm-roberta-large \
    --freeze-bert=False \
    --lstm-dim=-1 \
    --language=bangla \
    --seed=1 \
    --lr=5e-6 \
    --epoch=3 \
    --use-crf=False \
    --augment-type=all \
    --augment-rate=0.20 \
    --alpha-sub=0.4 \
    --alpha-del=0.4 \
    --loss-type=weighted-ce \
    --data-path=data \
    --save-path=out-weighted-ce

echo ""
echo "--- Standalone Evaluation (Weighted-CE) ---"

python src/test.py \
    --pretrained-model=xlm-roberta-large \
    --lstm-dim=-1 \
    --use-crf=False \
    --data-path=data/test \
    --weight-path=out-weighted-ce/weights.pt \
    --sequence-length=256 \
    --save-path=out-weighted-ce

echo ""
echo "✅ Experiment 3 complete. Results in out-weighted-ce/"

## Cell 6: 📊 Results Comparison Table

This cell parses all experiment logs and generates a side-by-side comparison table.

In [ ]:
import re
import os

PUNCTUATION_CLASSES = ['Comma', 'Period', 'Question', 'Exclamation', 'Overall']
TEST_SETS = ['test_news', 'test_asr', 'test_ref']

EXPERIMENTS = {
    'CE (Baseline)': 'Punctuation-Restoration-Bangla-Aug-Exp-2/out-ce/logs_test.txt',
    'Focal Loss': 'Punctuation-Restoration-Bangla-Aug-Exp-2/out-focal/logs_test.txt',
    'Weighted-CE': 'Punctuation-Restoration-Bangla-Aug-Exp-2/out-weighted-ce/logs_test.txt',
}


def parse_log_file(filepath):
    """Parse a test log file and extract per-test-set metrics."""
    if not os.path.exists(filepath):
        print(f"  ⚠️  Log file not found: {filepath}")
        return {}
    
    with open(filepath, 'r') as f:
        content = f.read()
    
    results = {}
    # Split by test set sections
    blocks = content.strip().split('\n\n')
    
    current_test_set = None
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        
        lines = block.strip().split('\n')
        
        # Check if first line is a test set name
        first_line = lines[0].strip()
        for ts in TEST_SETS:
            if ts in first_line:
                current_test_set = ts
                break
        
        # Parse precision, recall, f1, accuracy from the block
        precision_vals = None
        recall_vals = None
        f1_vals = None
        accuracy = None
        
        for line in lines:
            if line.startswith('Precision:'):
                nums = re.findall(r'[\d.]+', line)
                precision_vals = [float(x) for x in nums]
            elif line.startswith('Recall:'):
                nums = re.findall(r'[\d.]+', line)
                recall_vals = [float(x) for x in nums]
            elif line.startswith('F1 score:'):
                nums = re.findall(r'[\d.]+', line)
                f1_vals = [float(x) for x in nums]
            elif line.startswith('Accuracy:'):
                nums = re.findall(r'[\d.]+', line)
                if nums:
                    accuracy = float(nums[0])
        
        if current_test_set and f1_vals is not None:
            # f1_vals: [O, Comma, Period, Question, Exclamation, Overall]
            # We want indices 1-5 (skip O at index 0)
            results[current_test_set] = {
                'precision': precision_vals,
                'recall': recall_vals,
                'f1': f1_vals,
                'accuracy': accuracy
            }
    
    return results


def format_pct(val):
    """Format a 0-1 float as percentage string."""
    if val is None:
        return '  N/A  '
    return f'{val * 100:6.2f}%'


# Parse all experiments
all_results = {}
for exp_name, log_path in EXPERIMENTS.items():
    print(f"Parsing: {exp_name} ({log_path})")
    all_results[exp_name] = parse_log_file(log_path)

print()

# Print comparison tables
exp_names = list(EXPERIMENTS.keys())

for ts in TEST_SETS:
    ts_display = ts.replace('test_', '').upper()
    print(f"\n{'=' * 90}")
    print(f"  TEST SET: {ts_display}")
    print(f"{'=' * 90}")
    
    # Header
    header = f"{'Metric':<14}"
    for exp in exp_names:
        header += f" | {exp:>14}"
    print(header)
    print('-' * len(header))
    
    # Accuracy row
    row = f"{'Accuracy':<14}"
    for exp in exp_names:
        r = all_results.get(exp, {}).get(ts, {})
        acc = r.get('accuracy')
        row += f" | {format_pct(acc):>14}"
    print(row)
    print('-' * len(header))
    
    # F1 per class (indices 1=Comma, 2=Period, 3=Question, 4=Exclamation, 5=Overall)
    for i, cls_name in enumerate(PUNCTUATION_CLASSES):
        idx = i + 1  # skip index 0 (O class)
        row = f"{cls_name + ' F1':<14}"
        for exp in exp_names:
            r = all_results.get(exp, {}).get(ts, {})
            f1_vals = r.get('f1')
            val = f1_vals[idx] if f1_vals and len(f1_vals) > idx else None
            row += f" | {format_pct(val):>14}"
        
        # Highlight exclamation mark row
        if cls_name == 'Exclamation':
            row += '  ⬅️ KEY METRIC'
        print(row)
    
    print()

# Summary: delta table for exclamation F1
print(f"\n{'=' * 90}")
print(f"  SUMMARY: Exclamation Mark F1 Comparison")
print(f"{'=' * 90}")
header = f"{'Test Set':<14}"
for exp in exp_names:
    header += f" | {exp:>14}"
header += f" | {'Δ Focal':>10} | {'Δ W-CE':>10}"
print(header)
print('-' * len(header))

for ts in TEST_SETS:
    ts_display = ts.replace('test_', '').upper()
    row = f"{ts_display:<14}"
    
    baseline_f1 = None
    focal_f1 = None
    weighted_f1 = None
    
    for j, exp in enumerate(exp_names):
        r = all_results.get(exp, {}).get(ts, {})
        f1_vals = r.get('f1')
        val = f1_vals[4] if f1_vals and len(f1_vals) > 4 else None  # Exclamation = index 4
        row += f" | {format_pct(val):>14}"
        
        if j == 0:
            baseline_f1 = val
        elif j == 1:
            focal_f1 = val
        elif j == 2:
            weighted_f1 = val
    
    # Compute deltas
    if baseline_f1 is not None and focal_f1 is not None:
        delta_focal = (focal_f1 - baseline_f1) * 100
        delta_str = f"{delta_focal:+.2f}pp"
    else:
        delta_str = 'N/A'
    row += f" | {delta_str:>10}"
    
    if baseline_f1 is not None and weighted_f1 is not None:
        delta_wce = (weighted_f1 - baseline_f1) * 100
        delta_str = f"{delta_wce:+.2f}pp"
    else:
        delta_str = 'N/A'
    row += f" | {delta_str:>10}"
    
    print(row)

print()
print("📋 Copy this output for the paper revision.")
print("   Positive Δ = improvement over baseline CE.")
print("   Negative Δ = degradation vs baseline CE.")